## PROCESO ETL PARA REALIZAR LA LIMPIEZA Y CARGA EN EL DW

### LOS SCRIPTS SE ENCUENTRAN EN LA RUTA  /notebooks/Ejercicio_2/Scripts_BD/Script_Ejercicio2.sql

## LIBRERIAS PARA QUE EL CODIGO FUNCIONE CORRECTAMENTE

In [56]:
!pip install sqlalchemy psycopg2-binary
!pip install unidecode

## EXTRACCION

In [36]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
import unidecode

# Conexión a la base transaccional
user = "postgres"
password = "postgres"
host = "localhost"
port = "5432"
## Acceso a la base transaccional
db_transaccional = "postgres"

## Acceso al DW para escribir los nuevos datos
## aqui se coloco la misma db postgres, debido
## a que no se podia crear una nueva BD en el contenedor, pero
## cumple con su funcion, ya que solo debe cambiarse
## el nombre de la BD por DWH_MP3 para aplique la logica del DW
db_dw = "postgres"

# Crear engines
engine_transaccional = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db_transaccional}")
engine_dw = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db_dw}")

# Leer datos desde base transaccional
df_clientes = pd.read_sql("SELECT * FROM cliente", engine_transaccional)
df_productos = pd.read_sql("SELECT * FROM producto", engine_transaccional)
df_corresponsales = pd.read_sql("SELECT * FROM corresponsal", engine_transaccional)
df_ordenes = pd.read_sql("SELECT * FROM orden_compra", engine_transaccional)
df_detalles = pd.read_sql("SELECT * FROM detalle_orden", engine_transaccional)

## TRANSFORMACION

In [37]:
# LIMPIEZA DE DATOS
# Función de limpieza de texto
def limpiar_texto(texto):
    if pd.isnull(texto):
        return texto
    texto = str(texto).upper()
    texto = unidecode.unidecode(texto)
    texto = texto.replace('_', ' ').replace('-', ' ')
    return texto

# Función de redondeo superior a dos decimales
def redondear_superior(x):
    return np.ceil(x * 100) / 100 if pd.notnull(x) else x

# Clientes
for col in ['nombres', 'apellidos', 'correo_electronico', 'direccion']:
    df_clientes[col] = df_clientes[col].apply(limpiar_texto)
print(df_clientes.head(5))

# Convertir la columna 'fecha_pago' a tipo datetime
df_ordenes['fecha_pago'] = pd.to_datetime(df_ordenes['fecha_pago'])

   id_cliente      nombres     apellidos      cedula  \
0           1  JOSE ANDRES   MUNOZ PEREZ  0912345671   
1           2  MARIA JULIA   GOMEZ NUNEZ  0912345672   
2           3       CARLOS       RAMIREZ  0912345673   
3           4    ANA LUCIA      ZAMBRANO  0912345674   
4           5  LUIS MIGUEL  TORRES NAHUI  0912345675   

           correo_electronico              direccion    telefono  \
0      JOSE.MUNOZ@EXAMPLE.COM   AV. SIEMPRE VIVA 123  0987654321   
1     MARIA.GOMEZ@EXAMPLE.COM       CALLE 5 DE JUNIO  0987654322   
2  CARLOS.RAMIREZ@EXAMPLE.COM       AV. 10 DE AGOSTO  0987654323   
3    ANA.ZAMBRANO@EXAMPLE.COM    CDLA. LAS ORQUIDEAS  0987654324   
4     LUIS.TORRES@EXAMPLE.COM  CALLE BOLIVAR Y SUCRE  0987654325   

                   creado_en             actualizado_en  
0 2025-04-30 08:34:23.146983 2025-04-30 08:34:23.146983  
1 2025-04-30 08:34:23.146983 2025-04-30 08:34:23.146983  
2 2025-04-30 08:34:23.146983 2025-04-30 08:34:23.146983  
3 2025-04-30 08:34:23.

In [38]:
# Productos
# Limpiar el texto de las columnas 'nombre' y 'descripcion'
for col in ['nombre', 'descripcion']:
    df_productos[col] = df_productos[col].apply(limpiar_texto)

# Redondear el precio al inmediato superior
df_productos['precio_unit'] = df_productos['precio_unit'].apply(redondear_superior)
# Renombrar la columna 'nombre' a 'nombre_producto'
df_productos.rename(columns={'nombre': 'nombre_producto'}, inplace=True)
print(df_productos.head(5))

   id_producto                 nombre_producto  \
0            1            TELEVISOR LED 50" 4K   
1            2               LAPTOP HP ENVY 13   
2            3  AURICULARES INALAMBRICOS NTUNE   
3            4               ALTAVOZ SONY XB33   
4            5       RADIO DE AUTO PIONEER DEH   

                                         descripcion  precio_unit  disponible  \
0  TELEVISOR INTELIGENTE CON RESOLUCION ULTRA HD ...       550.00        True   
1  PORTATIL ULTRADELGADA CON PROCESADOR INTEL I7 ...       899.51        True   
2  AURICULARES BLUETOOTH CON CANCELACION DE RUIDO...       130.00        True   
3  PARLANTE PORTATIL RESISTENTE AL AGUA CON BAJOS...       149.13        True   
4  REPRODUCTOR DE MUSICA PARA AUTOMOVIL CON USB Y...        90.00        True   

                   creado_en             actualizado_en  
0 2025-04-30 08:34:23.150953 2025-04-30 08:34:23.150953  
1 2025-04-30 08:34:23.150953 2025-04-30 08:34:23.150953  
2 2025-04-30 08:34:23.150953 2025-04-3

In [39]:
# Tiempo
df_ordenes['fecha_pago'] = pd.to_datetime(df_ordenes['fecha_pago'], errors='coerce')
df_ordenes['fecha_pago_aux'] = pd.to_datetime(df_ordenes['fecha_pago'], errors='coerce')
df_ordenes['fecha_pago'] = df_ordenes['fecha_pago'].dt.strftime('%Y-%m-%d')

df_tiempo = pd.DataFrame()
df_tiempo['fecha'] = df_ordenes['fecha_pago_aux']
df_tiempo['fecha_aux'] = df_ordenes['fecha_pago']
df_tiempo['dia'] = df_tiempo['fecha'].dt.day
df_tiempo['mes'] = df_tiempo['fecha'].dt.month
df_tiempo['anio'] = df_tiempo['fecha'].dt.year
df_tiempo['nombre_mes'] = df_tiempo['fecha'].dt.month_name()
df_tiempo = df_tiempo.drop(columns=['fecha'])
df_tiempo.rename(columns={'fecha_aux': 'fecha'}, inplace=True)
df_tiempo = df_tiempo.drop_duplicates(subset=['fecha'])
df_tiempo['id_fecha'] = range(1, len(df_tiempo) + 1)
df_ordenes.rename(columns={'fecha_pago_aux': 'fecha'}, inplace=True)
df_ordenes = df_ordenes.drop(columns=['fecha'])
df_ordenes.rename(columns={'fecha_pago': 'fecha'}, inplace=True)
print(df_tiempo.head(5))

         fecha  dia  mes  anio nombre_mes  id_fecha
0   2025-04-29   29    4  2025      April         1
7   2025-04-30   30    4  2025      April         2
19  2025-03-30   30    3  2025      March         3
20  2025-04-28   28    4  2025      April         4


In [40]:
# Corresponsales
# Limpiar el texto de las columnas 'nombre' y 'ubicacion'
for col in ['nombre', 'ubicacion']:
    df_corresponsales[col] = df_corresponsales[col].apply(limpiar_texto)

# Renombrar la columna 'nombre' a 'nombre_corresponsal'
df_corresponsales.rename(columns={'nombre': 'nombre_corresponsal'}, inplace=True)
print(df_corresponsales.head(5))

   id_corresponsal         nombre_corresponsal  \
0                1             TECNOLOGIA NANA   
1                2                 AUDIO MUNDO   
2                3               ELECTRO HOGAR   
3                4                   AUTO ZONA   
4                5  MULTISERVICIOS LOPEZ PEREZ   

                                  ubicacion    telefono  \
0  AV. BOLIVAR E4 27 Y GARCIA MORENO, QUITO  0998765432   
1      CALLE 10 DE AGOSTO Y TARQUI   CUENCA  0987654321   
2       AV. DE LOS SHYRIS Y PORTUGAL, QUITO  0967123456   
3        PANAMERICANA NORTE KM 15.5, IBARRA  0976543210   
4    AV. LA PRENSA Y RAMIREZ DAVALOS, QUITO  0954321098   

                   creado_en  
0 2025-04-30 08:34:23.153770  
1 2025-04-30 08:34:23.153770  
2 2025-04-30 08:34:23.153770  
3 2025-04-30 08:34:23.153770  
4 2025-04-30 08:34:23.153770  


In [41]:
# Detalles de orden
df_detalles['precio_unitario'] = df_detalles['precio_unitario'].apply(redondear_superior)
df_detalles['subtotal'] = df_detalles['subtotal'].apply(redondear_superior)
print(df_detalles.head(5))

   id_detalle  id_orden  id_producto  cantidad  precio_unitario  subtotal
0           1         1            1         1           550.00    550.00
1           2         1            2         2           899.51   1799.02
2           3         2            3         1           130.00    130.00
3           4         2            4         2           149.13    298.25
4           5         3            5         1            90.00     90.00


## CARGA

In [42]:
# Tabla dim_producto (productos)
df_productos[['id_producto', 'nombre_producto', 'descripcion']].to_sql(
    "dim_producto", engine_dw, if_exists="append", index=False
)
print(df_productos.head(5))

   id_producto                 nombre_producto  \
0            1            TELEVISOR LED 50" 4K   
1            2               LAPTOP HP ENVY 13   
2            3  AURICULARES INALAMBRICOS NTUNE   
3            4               ALTAVOZ SONY XB33   
4            5       RADIO DE AUTO PIONEER DEH   

                                         descripcion  precio_unit  disponible  \
0  TELEVISOR INTELIGENTE CON RESOLUCION ULTRA HD ...       550.00        True   
1  PORTATIL ULTRADELGADA CON PROCESADOR INTEL I7 ...       899.51        True   
2  AURICULARES BLUETOOTH CON CANCELACION DE RUIDO...       130.00        True   
3  PARLANTE PORTATIL RESISTENTE AL AGUA CON BAJOS...       149.13        True   
4  REPRODUCTOR DE MUSICA PARA AUTOMOVIL CON USB Y...        90.00        True   

                   creado_en             actualizado_en  
0 2025-04-30 08:34:23.150953 2025-04-30 08:34:23.150953  
1 2025-04-30 08:34:23.150953 2025-04-30 08:34:23.150953  
2 2025-04-30 08:34:23.150953 2025-04-3

In [43]:
# Para la tabla dim_tiempo (tiempo)
df_tiempo[['id_fecha', 'fecha', 'dia', 'mes', 'anio', 'nombre_mes']].to_sql(
    "dim_tiempo", engine_dw, if_exists="append", index=False
)
print(df_tiempo.head(5))

         fecha  dia  mes  anio nombre_mes  id_fecha
0   2025-04-29   29    4  2025      April         1
7   2025-04-30   30    4  2025      April         2
19  2025-03-30   30    3  2025      March         3
20  2025-04-28   28    4  2025      April         4


In [44]:
# Para la tabla dim_cliente
df_clientes[['id_cliente', 'nombres', 'apellidos', 'cedula', 'correo_electronico']].to_sql(
    "dim_cliente", engine_dw, if_exists="append", index=False
)
print(df_clientes.head(5))

   id_cliente      nombres     apellidos      cedula  \
0           1  JOSE ANDRES   MUNOZ PEREZ  0912345671   
1           2  MARIA JULIA   GOMEZ NUNEZ  0912345672   
2           3       CARLOS       RAMIREZ  0912345673   
3           4    ANA LUCIA      ZAMBRANO  0912345674   
4           5  LUIS MIGUEL  TORRES NAHUI  0912345675   

           correo_electronico              direccion    telefono  \
0      JOSE.MUNOZ@EXAMPLE.COM   AV. SIEMPRE VIVA 123  0987654321   
1     MARIA.GOMEZ@EXAMPLE.COM       CALLE 5 DE JUNIO  0987654322   
2  CARLOS.RAMIREZ@EXAMPLE.COM       AV. 10 DE AGOSTO  0987654323   
3    ANA.ZAMBRANO@EXAMPLE.COM    CDLA. LAS ORQUIDEAS  0987654324   
4     LUIS.TORRES@EXAMPLE.COM  CALLE BOLIVAR Y SUCRE  0987654325   

                   creado_en             actualizado_en  
0 2025-04-30 08:34:23.146983 2025-04-30 08:34:23.146983  
1 2025-04-30 08:34:23.146983 2025-04-30 08:34:23.146983  
2 2025-04-30 08:34:23.146983 2025-04-30 08:34:23.146983  
3 2025-04-30 08:34:23.

In [45]:
# Para la tabla dim_corresponsal (corresponsales)
df_corresponsales[['id_corresponsal', 'nombre_corresponsal', 'ubicacion']].to_sql(
    "dim_corresponsal", engine_dw, if_exists="append", index=False
)
print(df_corresponsales.head(5))

   id_corresponsal         nombre_corresponsal  \
0                1             TECNOLOGIA NANA   
1                2                 AUDIO MUNDO   
2                3               ELECTRO HOGAR   
3                4                   AUTO ZONA   
4                5  MULTISERVICIOS LOPEZ PEREZ   

                                  ubicacion    telefono  \
0  AV. BOLIVAR E4 27 Y GARCIA MORENO, QUITO  0998765432   
1      CALLE 10 DE AGOSTO Y TARQUI   CUENCA  0987654321   
2       AV. DE LOS SHYRIS Y PORTUGAL, QUITO  0967123456   
3        PANAMERICANA NORTE KM 15.5, IBARRA  0976543210   
4    AV. LA PRENSA Y RAMIREZ DAVALOS, QUITO  0954321098   

                   creado_en  
0 2025-04-30 08:34:23.153770  
1 2025-04-30 08:34:23.153770  
2 2025-04-30 08:34:23.153770  
3 2025-04-30 08:34:23.153770  
4 2025-04-30 08:34:23.153770  


In [46]:
# Hecho - Para la tabla hecho_ventas
df_ventas = df_detalles.merge(df_ordenes, on="id_orden") 
df_ventas_2 = df_ventas.merge(df_tiempo, on="fecha")
df_ventas_2['total_pagado'] = df_ventas_2['cantidad'] * df_ventas_2['precio_unitario']  
df_ventas_2 = df_ventas_2.drop(columns=['mes', 'anio', 'nombre_mes', 'dia'])
# Insertar en hecho_ventas
df_ventas_2[['id_fecha', 'id_producto', 'id_cliente', 'id_corresponsal', 'cantidad', 'total_pagado']].to_sql(
    "hecho_ventas", engine_dw, if_exists="append", index=False
)
print(df_ventas_2.head(5))

   id_detalle  id_orden  id_producto  cantidad  precio_unitario  subtotal  \
0           1         1            1         1           550.00    550.00   
1           2         1            2         2           899.51   1799.02   
2           3         2            3         1           130.00    130.00   
3           4         2            4         2           149.13    298.25   
4           5         3            5         1            90.00     90.00   

   id_cliente         fecha_orden numero_pago  id_corresponsal  estado  \
0           1 2025-04-29 10:30:00     ORD0001                1  PAGADA   
1           1 2025-04-29 10:30:00     ORD0001                1  PAGADA   
2           1 2025-04-29 12:15:00     ORD0002                2  PAGADA   
3           1 2025-04-29 12:15:00     ORD0002                2  PAGADA   
4           2 2025-04-29 14:00:00     ORD0003                3  PAGADA   

        fecha                                     comentario  id_fecha  \
0  2025-04-29  Com